# 01. Data exploration

The face-crop dataset built by `ml/scripts/prepare_data.py` from four public sources. Run that script first (see `docs/DATASETS.md`).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "app").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
DATA = ROOT / "data" / "crops"
REPORTS = ROOT / "reports"


In [2]:
df = pd.read_csv(DATA / 'manifest.csv')
print(f'{len(df):,} face crops')
df.head()

4,760 face crops


,path,source,split,label,group,video,face_found
0,ffhq_stylegan/train/fake/train_0.png,ffhq_stylegan,train,1,ffhq_train_0,NaN,1
1,ffhq_stylegan/train/real/train_1.png,ffhq_stylegan,train,0,ffhq_train_1,NaN,1
2,ffhq_stylegan/train/fake/train_2.png,ffhq_stylegan,train,1,ffhq_train_2,NaN,1
3,ffhq_stylegan/train/fake/train_3.png,ffhq_stylegan,train,1,ffhq_train_3,NaN,1
4,ffhq_stylegan/train/fake/train_4.png,ffhq_stylegan,train,1,ffhq_train_4,NaN,1


## Balance per source, split and label

Every source is balanced between real and fake, so accuracy is meaningful per source.

In [3]:
pd.crosstab([df.source, df.split], df.label.map({0: 'real', 1: 'fake'}))

label                fake  real
source        split            
celebdf       test    160   160
              train   480   480
dfd           test    100   100
              train   240   240
dff           test    150   150
              train   400   400
ffhq_stylegan test    250   250
              train   600   600

## Face detection coverage

Images without a detected face are kept and analyzed whole (the same thing the app does). Video frames without a face were skipped during extraction.

In [4]:
df.groupby('source').face_found.mean().round(3).rename('share with a detected face')

source
celebdf          1.000
dfd              1.000
dff              0.928
ffhq_stylegan    1.000
Name: share with a detected face, dtype: float64

## No leakage between splits

Identities (Celeb-DF, DeepFakeFace), actors (DFD) and source videos never appear in both train and test.

In [5]:
for src, sub in df.groupby('source'):
    tr = set(sub[sub.split == 'train'].group.astype(str)); te = set(sub[sub.split == 'test'].group.astype(str))
    print(f'{src:14s} train groups={len(tr):5d} test groups={len(te):5d} overlap={len(tr & te)}')

celebdf        train groups=  105 test groups=   50 overlap=0
dfd            train groups=   20 test groups=    8 overlap=0
dff            train groups=  800 test groups=  300 overlap=0
ffhq_stylegan  train groups= 1200 test groups=  500 overlap=0


## What the model sees

Top row real, bottom row fake, one column pair per source.

In [6]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
rng = np.random.default_rng(3)
for c, src in enumerate(['ffhq_stylegan', 'dff', 'celebdf', 'dfd']):
    for lab in (0, 1):
        pool = df[(df.source == src) & (df.label == lab)]
        for k in range(2):
            ax = axes[lab, c * 2 + k]
            ax.imshow(Image.open(DATA / pool.iloc[rng.integers(len(pool))].path)); ax.axis('off')
            if k == 0: ax.set_title(f"{src}\n{'fake' if lab else 'real'}", fontsize=9)
plt.tight_layout()

Face images from the datasets are not stored in this notebook (dataset licenses). Run it locally after prepare_data.py to see them.


## Crop sizes

Video crops are smaller (faces are smaller in frame), which is one reason video detection is harder.

In [7]:
sizes = [Image.open(DATA / p).size[0] for p in df.path]
df['side_px'] = sizes
df.groupby('source').side_px.describe()[['mean', '25%', '50%', '75%']].round(0)

,mean,25%,50%,75%
source,,,,
celebdf,278.0,234.0,271.0,315.0
dfd,316.0,232.0,312.0,425.0
dff,257.0,153.0,240.0,343.0
ffhq_stylegan,254.0,256.0,256.0,256.0
